# 02 · Preparación y construcción de variables (Fase 4)

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_columns", 30)
from src.config import *

In [2]:
from src.data import build_dataset
df = build_dataset.main()
print(df.shape)

Índice de origen:
  varianza explicada 1er componente: 21.6%
Índice de destino:
  varianza explicada 1er componente: 15.6%


(17843, 145)


## Índices de riqueza
- **Origen:** 45 bienes y servicios del hogar a los 14 años, más piso de tierra, autos y cuartos.
- **Destino:** 44 bienes y servicios del hogar actual, más piso de tierra y autos.
- Cada índice es el **primer componente principal** (PCA), que resume muchas variables sí/no en un solo número.
- Después se calculan **quintiles ponderados dentro de cada cohorte**: Q1 = el 20% más pobre de su generación.

In [3]:
cargas = pd.read_csv(TABLES / "cargas_origen.csv", index_col=0)
display(Markdown("**Variables que más pesan en el índice de origen**")); display(cargas.head(10))

**Variables que más pesan en el índice de origen**

,carga
o_p31b,0.225
o_p31c,0.214
o_p26c,0.212
o_p26d,0.212
o_p29a,0.209
o_p31a,0.198
o_p31d,0.193
o_p26a,0.193
o_p31e,0.192
o_p29f,0.192


## Validación de los índices
Un índice bien construido debe parecerse a otras medidas de bienestar que **no** se usaron para construirlo.

In [4]:
val = pd.DataFrame({
  "Comparación": ["Índice destino vs log(ingreso per cápita)", "Índice origen vs escalera subjetiva a los 14 (1-10)", "Índice destino vs escalera subjetiva hoy", "Índice origen vs índice destino"],
  "Correlación": [np.corrcoef(df.indice_destino, np.log(df.ingreso_pc))[0,1], np.corrcoef(df.indice_origen, df.escalera_origen_subj)[0,1],
                  np.corrcoef(df.indice_destino, df.escalera_actual_subj)[0,1], np.corrcoef(df.indice_origen, df.indice_destino)[0,1]]}).round(2)
val.to_csv(TABLES / "validacion_indices.csv", index=False); display(val)

,Comparación,Correlación
0,Índice destino vs log(ingreso per cápita),0.61
1,Índice origen vs escalera subjetiva a los 14 (...,0.50
2,Índice destino vs escalera subjetiva hoy,0.40
3,Índice origen vs índice destino,0.62


In [5]:
w = df.factor
dist = df.groupby("movilidad").factor.sum().div(w.sum()).mul(100).round(1)
display(Markdown("**Variable objetivo secundaria: movilidad (% ponderado)**")); display(dist.to_frame("%"))
display(Markdown(f"**Variable objetivo principal:** {df.sigue_q1.notna().sum():,} personas nacieron en Q1; el {np.average(df.sigue_q1.dropna(), weights=w[df.sigue_q1.notna()])*100:.1f}% sigue en Q1."))

**Variable objetivo secundaria: movilidad (% ponderado)**

,%
movilidad,
Ascenso,30.100000
Descenso,31.200001
Permanencia,38.700001


**Variable objetivo principal:** 2,674 personas nacieron en Q1; el 51.1% sigue en Q1.

## Partición y prevención de *leakage*
- Muestra: 2,674 personas que nacieron en Q1. Se divide en **80% entrenamiento y 20% prueba**, estratificado y con semilla fija (42).
- Las variables de destino (bienes actuales, escolaridad propia, ocupación, ingreso) **no** entran como predictores del modelo principal, porque son parte de la respuesta.
- Los faltantes numéricos se imputan con la mediana **dentro** del pipeline: se ajusta solo con los datos de entrenamiento.

**Estado de la fase:** PROPUESTA — falta la revisión humana (regla de avance del roadmap: APROBADA / APROBADA CON CAMBIOS / REABRIR / BLOQUEADA).